# 工具使用和函数调用

Toolformer开启了自监督的工具标注。Berkeley Function Calling Leaderboard（BFCL）V4 把评测重心从「单次 JSON 调对没有」挪到智能体式工具使用。单轮大体已饱和；记忆、动态决策、长跨度工具链仍是主战场。

## 问题描述

早期的工具使用问的是：模型是否能预测出一个正确的工具调用？现在的工具使用问的是：模型的工具链是否可以跨40步、带记忆、带部分观察结果、失败时修复、不幻觉出不存在的工具。

Toolformer建立的基线：模型可以通过自监督学习什么时候调用工具。BFCL V4定义了2026年的评估目标。两者之间的差距就是生产级agent需要跨越的。

## 基本概念

### Toolformer

目标：不用大量人工 Function Calling 标注，也能让模型学会「何时调工具、调哪个、参数怎么填」。

做法是自监督流水线，四步：

1. **候选标注**  
   在预训练文本的某些位置，让模型采样可能的 API 调用（工具名 + 参数），得到许多候选插入。

2. **真执行**  
   对每个候选实际调用对应 API，把返回值写回序列（调用 + 结果一起出现在上下文里）。

3. **用下一 token 损失过滤**  
   比较「不插工具」与「插入工具结果」两种前缀下，对**后续真实文本**的预测 loss。  
   只有工具结果能明显降低后续 loss 的候选才保留——说明这次调用真的提供了有用信息；多余或错误的调用丢掉。

4. **过滤语料上微调**  
   用留下的「文本 + 有用工具调用 + 真实结果」样本微调，模型据此学会工具使用时机与格式。

一句话：候选调用 → 执行 → loss 下降才留下 → 微调。这是单点、局部有用工具的基线；多轮、记忆、长链与防幻觉仍要靠 BFCL 一类更难设定来衡量。

### BFCL V4

BFCL（Berkeley Function Calling Leaderboard）由 Gorilla / Berkeley 团队维护，用 **AST 匹配或状态转移** 做确定性打分（尽量不用 LLM-as-judge），方便复现。版本演进大致是：

| 版本 | 侧重点 |
|------|--------|
| V1 | 单轮：simple / multiple / parallel；AST 评结构对不对 |
| V2 | 加入社区/企业贡献的 Live 函数与可执行评测 |
| V3 | 多轮、多步；可追问、补全参数 |
| V4 | **整体智能体评测**：网页搜索、记忆、格式敏感性等 |

#### 总分怎么加权

```text
Overall ≈ Agentic×40% + Multi-Turn×30% + Live×10% + Non-Live×10% + Hallucination×10%
```

| 权重 | 类别 | 考什么 |
|------|------|--------|
| **40%** | **Agentic** | 更像真 agent：联网搜索、跨轮记忆读写、schema/提示格式一变是否还稳（format sensitivity） |
| **30%** | **Multi-Turn** | 多轮对话里持续正确用工具；含缺函数、缺参数、长上下文等——该追问时别硬调 |
| **10%** | **Live** | 真实/社区贡献 API，可执行路径 |
| **10%** | **Non-Live** | 专家策展的经典单轮题（simple / parallel 等） |
| **10%** | **Hallucination** | **该不调就不调**：请求与可用工具无关时要拒绝，别臆造工具或乱调 |

经典「单次吐对 JSON」只占 Live+Non-Live 合计约 **20%**；**70%** 压在 Agentic + Multi-Turn。信号是：单轮对了也不再代表生产可用。

#### 和 Toolformer 的关系

- Toolformer：训练侧——如何**造**工具使用能力（自监督标注）。
- BFCL V4：评测侧——2026 年什么叫「工具用得好」（尤其 agent 行为）。

中间缺口：记忆、动态决策（何时不用工具）、长视野链路与防幻觉。

### Tool Schema

每个模型供应商都有一个Schema，它们在细节上不同，但是形状是类似的。
```json
name: string
descrition: string (what it does, when to use it)
input_schema: JSON Schema (properties, required, types, enums)
```

### 参数调用

不要相信任何的tool call，需要进行验证。
- 数据类型纠正。
- 枚举验证。
- 必要字段验证。
- 格式验证。

每个验证失败都应该返回结构化的观察结果，好让模型重试的时候使用正确的形态。



# 开始编码

对应本章核心：**Toolformer 自监督过滤**、**Tool Schema + 参数校验**、**BFCL 式确定性打分（AST / 拒调）**。  
先用玩具跑通「候选 → 执行 → loss 过滤」；再用 **PyTorch 小 LM** 演示下一 token 损失；最后用 **LangChain + DeepSeek** 做原生 Function Calling。


## 1. 教学玩具：Toolformer 过滤 + Schema 校验

- 候选 API → 真执行 → 用「后续文本」的 next-token loss 决定去留（Toolformer 思想）。
- 校验失败返回**结构化观察**，供重试（笔记：不要盲信 tool call）。


In [ ]:
from __future__ import annotations

import json
import math
import re
from dataclasses import dataclass, field
from typing import Any, Callable, Literal

from typing_extensions import TypedDict


# ---------------------------------------------------------------------------
# Schema + 校验（BFCL / 生产 agent 共用习惯）
# ---------------------------------------------------------------------------


class ToolParamSpec(TypedDict, total=False):
    """单个参数的 JSON-Schema 风格描述。"""

    type: Literal["string", "number", "integer", "boolean"]
    enum: list[Any]
    description: str


class ToolSchema(TypedDict):
    """工具注册用的最小 Schema。"""

    name: str
    description: str
    parameters: dict[str, ToolParamSpec]
    required: list[str]


@dataclass
class ValidationIssue:
    """一次校验失败项。"""

    code: str
    field: str
    message: str


@dataclass
class ValidationResult:
    """参数校验结果。"""

    ok: bool
    cleaned: dict[str, Any] = field(default_factory=dict)
    issues: list[ValidationIssue] = field(default_factory=list)

    def as_observation(self) -> str:
        """
        Returns:
            observation: 结构化错误观察（模型可读 JSON）。
        """
        if self.ok:
            return json.dumps({"status": "ok", "args": self.cleaned}, ensure_ascii=False)
        payload = {
            "status": "validation_error",
            "issues": [
                {"code": i.code, "field": i.field, "message": i.message} for i in self.issues
            ],
            "hint": "Fix args and retry with the same tool name.",
        }
        return json.dumps(payload, ensure_ascii=False)


def validate_tool_args(schema: ToolSchema, raw_args: dict[str, Any]) -> ValidationResult:
    """
    校验工具参数：必填、类型纠正、枚举。

    Args:
        schema: 工具 Schema。
        raw_args: 模型给出的原始参数。

    Returns:
        result: ``ok`` 时含 ``cleaned``；失败时含 ``issues``。
    """
    issues: list[ValidationIssue] = []
    cleaned: dict[str, Any] = {}
    params = schema["parameters"]
    required = set(schema.get("required") or [])

    for key in required:
        if key not in raw_args or raw_args[key] is None or raw_args[key] == "":
            issues.append(
                ValidationIssue("missing_required", key, f"required field '{key}' is missing")
            )

    for key, value in raw_args.items():
        if key not in params:
            issues.append(ValidationIssue("unknown_field", key, f"unknown field '{key}'"))
            continue
        spec = params[key]
        expected = spec.get("type", "string")
        coerced: Any = value
        try:
            if expected == "string":
                coerced = str(value)
            elif expected == "integer":
                if isinstance(value, bool):
                    raise TypeError("bool is not integer")
                coerced = int(value)
            elif expected == "number":
                if isinstance(value, bool):
                    raise TypeError("bool is not number")
                coerced = float(value)
            elif expected == "boolean":
                if isinstance(value, bool):
                    coerced = value
                elif str(value).lower() in {"true", "1", "yes"}:
                    coerced = True
                elif str(value).lower() in {"false", "0", "no"}:
                    coerced = False
                else:
                    raise TypeError(f"cannot coerce {value!r} to bool")
        except (TypeError, ValueError) as e:
            issues.append(
                ValidationIssue("type_error", key, f"expected {expected}, got {value!r} ({e})")
            )
            continue

        enum = spec.get("enum")
        if enum is not None and coerced not in enum:
            issues.append(
                ValidationIssue("enum_error", key, f"value {coerced!r} not in {enum}")
            )
            continue
        cleaned[key] = coerced

    return ValidationResult(ok=len(issues) == 0, cleaned=cleaned, issues=issues)


def calculator(expr: str) -> str:
    """
    安全算术求值。

    Args:
        expr: 如 ``\"(3+5)*2\"``。

    Returns:
        result: 数值字符串或 ``Error: ...``。
    """
    allowed = set("0123456789+-*/(). ")
    if not set(expr).issubset(allowed):
        return "Error: illegal characters in expr"
    try:
        return str(eval(expr, {"__builtins__": {}}, {}))  # noqa: S307 — 玩具沙箱
    except Exception as e:
        return f"Error: {type(e).__name__}: {e}"


CALCULATOR_SCHEMA: ToolSchema = {
    "name": "calculator",
    "description": "Evaluate arithmetic. Use when the user needs a numeric result.",
    "parameters": {
        "expr": {"type": "string", "description": "Arithmetic expression"},
    },
    "required": ["expr"],
}

UNIT_CONVERT_SCHEMA: ToolSchema = {
    "name": "unit_convert",
    "description": "Convert between meters and kilometers.",
    "parameters": {
        "value": {"type": "number", "description": "Numeric magnitude"},
        "from_unit": {
            "type": "string",
            "enum": ["m", "km"],
            "description": "Source unit",
        },
        "to_unit": {
            "type": "string",
            "enum": ["m", "km"],
            "description": "Target unit",
        },
    },
    "required": ["value", "from_unit", "to_unit"],
}


def unit_convert(value: float, from_unit: str, to_unit: str) -> str:
    """
    Args:
        value: 数值。
        from_unit: ``m`` 或 ``km``。
        to_unit: ``m`` 或 ``km``。

    Returns:
        result: 转换结果字符串。
    """
    meters = value if from_unit == "m" else value * 1000.0
    out = meters if to_unit == "m" else meters / 1000.0
    return f"{out:g} {to_unit}"


TOOL_IMPL: dict[str, Callable[..., str]] = {
    "calculator": calculator,
    "unit_convert": unit_convert,
}
TOOL_SCHEMAS: dict[str, ToolSchema] = {
    "calculator": CALCULATOR_SCHEMA,
    "unit_convert": UNIT_CONVERT_SCHEMA,
}


def dispatch_validated(name: str, raw_args: dict[str, Any]) -> str:
    """
    Schema 校验后再执行；失败返回结构化观察。

    Args:
        name: 工具名。
        raw_args: 原始参数。

    Returns:
        observation: 成功结果或 validation / unknown 观察。
    """
    schema = TOOL_SCHEMAS.get(name)
    if schema is None:
        return json.dumps(
            {
                "status": "hallucinated_tool",
                "name": name,
                "available": sorted(TOOL_SCHEMAS),
                "hint": "Do not invent tools. Answer without a call or pick an available name.",
            },
            ensure_ascii=False,
        )
    checked = validate_tool_args(schema, raw_args)
    if not checked.ok:
        return checked.as_observation()
    fn = TOOL_IMPL[name]
    try:
        return fn(**checked.cleaned)
    except Exception as e:
        return f"Error: {type(e).__name__}: {e}"


# ---------------------------------------------------------------------------
# Toolformer 玩具：候选 → 执行 → 用 bag 语言模型的 NLL 过滤
# ---------------------------------------------------------------------------


@dataclass
class ApiCandidate:
    """一次候选工具插入。"""

    name: str
    args: dict[str, Any]
    span: tuple[int, int]  # 插入点前后字符索引（相对全文）


@dataclass
class FilteredExample:
    """通过 loss 过滤后保留的训练样本。"""

    prefix: str
    call_text: str
    tool_result: str
    suffix: str
    loss_without: float
    loss_with: float


class TinyBagLM:
    """
    极简 bag 语言模型：用训练语料上的 unigram/bigram 统计估计后缀 NLL。
    教学用，代替真实 Transformer 的 next-token loss。
    """

    def __init__(self) -> None:
        self.unigram: dict[str, float] = {}
        self.bigram: dict[tuple[str, str], float] = {}
        self.vocab: set[str] = set()

    @staticmethod
    def tokenize(text: str) -> list[str]:
        """
        Args:
            text: 任意文本。

        Returns:
            tokens: 小写单词列表。
        """
        return re.findall(r"[a-z0-9.]+", text.lower())

    def fit(self, corpus: list[str]) -> None:
        """
        Args:
            corpus: 纯文本语料行。
        """
        from collections import Counter

        uni: Counter[str] = Counter()
        bi: Counter[tuple[str, str]] = Counter()
        for line in corpus:
            toks = self.tokenize(line)
            self.vocab.update(toks)
            uni.update(toks)
            bi.update(zip(toks, toks[1:]))
        n = sum(uni.values()) or 1
        self.unigram = {w: c / n for w, c in uni.items()}
        # P(w2|w1) ≈ count(w1,w2)/count(w1)
        self.bigram = {}
        for (w1, w2), c in bi.items():
            self.bigram[(w1, w2)] = c / max(uni[w1], 1)

    def nll(self, context: str, target: str) -> float:
        """
        估计在 ``context`` 条件下生成 ``target`` 的平均负对数似然（越小越好）。

        教学技巧：若工具结果里的数字/单位已出现在 ``context``，则对应
        target token 的概率被抬高——模拟「观察降低后续 loss」。

        Args:
            context: 前缀（可含工具结果）。
            target: 后续真实文本。

        Returns:
            nll: 平均 NLL；空 target 返回 ``inf``。
        """
        ctx = self.tokenize(context)
        tgt = self.tokenize(target)
        if not tgt:
            return float("inf")
        ctx_set = set(ctx)
        prev = ctx[-1] if ctx else "<bos>"
        total = 0.0
        for tok in tgt:
            p_bi = self.bigram.get((prev, tok), 0.0)
            p_uni = self.unigram.get(tok, 1e-6)
            p = 0.9 * p_bi + 0.1 * p_uni if p_bi > 0 else p_uni
            # 工具结果已在上下文中 → 该 token 更易预测（Toolformer 信号）
            if tok in ctx_set and re.fullmatch(r"[0-9.]+|[a-z]+", tok):
                p = max(p, 0.55)
            total += -math.log(max(p, 1e-9))
            prev = tok
        return total / len(tgt)


def format_api_call(name: str, args: dict[str, Any], result: str) -> str:
    """
    Args:
        name: 工具名。
        args: 已校验参数。
        result: 执行结果。

    Returns:
        call_text: 插入正文的 API 片段。
    """
    return f"[API:{name}({json.dumps(args, ensure_ascii=False)})→{result}]"


def propose_candidates(text: str) -> list[ApiCandidate]:
    """
    启发式候选标注（真实 Toolformer 由模型采样）。

    Args:
        text: 预训练式句子，如 ``The distance is 2000 m which is 2 km.``。

    Returns:
        candidates: 可能的 API 插入。
    """
    cands: list[ApiCandidate] = []
    # 算术：找到 "X + Y = Z" 或 "is N" 模式旁插 calculator
    m = re.search(r"(\d+)\s*\+\s*(\d+)\s*=\s*(\d+)", text)
    if m:
        a, b, _ = m.group(1), m.group(2), m.group(3)
        cands.append(
            ApiCandidate("calculator", {"expr": f"{a}+{b}"}, (m.start(), m.start()))
        )
        cands.append(
            ApiCandidate("calculator", {"expr": f"{a}-{b}"}, (m.start(), m.start()))
        )  # 坏候选
    m2 = re.search(r"(\d+(?:\.\d+)?)\s*m\b.*?\b(\d+(?:\.\d+)?)\s*km\b", text, re.I)
    if m2:
        cands.append(
            ApiCandidate(
                "unit_convert",
                {"value": float(m2.group(1)), "from_unit": "m", "to_unit": "km"},
                (m2.start(), m2.start()),
            )
        )
        cands.append(
            ApiCandidate(
                "unit_convert",
                {"value": float(m2.group(1)), "from_unit": "km", "to_unit": "m"},
                (m2.start(), m2.start()),
            )
        )  # 坏候选：方向反了
    return cands


def toolformer_filter(
    text: str,
    lm: TinyBagLM,
    *,
    min_improvement: float = 0.05,
) -> list[FilteredExample]:
    """
    Toolformer 核心：只保留「插入工具结果后后缀 NLL 明显下降」的候选。

    Args:
        text: 完整句子。
        lm: 已 ``fit`` 的 bag LM。
        min_improvement: ``loss_without - loss_with`` 下限。

    Returns:
        kept: 通过过滤的样本。
    """
    kept: list[FilteredExample] = []
    for cand in propose_candidates(text):
        obs = dispatch_validated(cand.name, cand.args)
        if obs.startswith("{") and '"status"' in obs and "ok" not in obs[:40]:
            # 校验/幻觉失败不当作有用结果
            continue
        if obs.startswith("Error:"):
            continue
        i = cand.span[0]
        prefix, suffix = text[:i], text[i:]
        call_text = format_api_call(cand.name, cand.args, obs)
        loss_wo = lm.nll(prefix, suffix)
        loss_w = lm.nll(prefix + call_text, suffix)
        if loss_wo - loss_w >= min_improvement:
            kept.append(
                FilteredExample(
                    prefix=prefix,
                    call_text=call_text,
                    tool_result=obs,
                    suffix=suffix,
                    loss_without=loss_wo,
                    loss_with=loss_w,
                )
            )
    return kept


# ---------------------------------------------------------------------------
# BFCL 风格：单轮 AST 匹配 + 拒调（Hallucination）打分
# ---------------------------------------------------------------------------


@dataclass
class ExpectedCall:
    """期望的一次函数调用（Non-Live / AST）。"""

    name: str
    args: dict[str, Any]


def ast_match(pred: ExpectedCall | None, gold: ExpectedCall | None) -> bool:
    """
    简化 AST 匹配：工具名 + 参数字典全等（数值容差）。

    Args:
        pred: 模型预测；``None`` 表示拒调。
        gold: 标准答案；``None`` 表示应拒调。

    Returns:
        match: 是否判对。
    """
    if pred is None and gold is None:
        return True
    if pred is None or gold is None:
        return False
    if pred.name != gold.name:
        return False
    if set(pred.args) != set(gold.args):
        return False
    for k, gv in gold.args.items():
        pv = pred.args[k]
        if isinstance(gv, (int, float)) and isinstance(pv, (int, float)):
            if abs(float(pv) - float(gv)) > 1e-6:
                return False
        elif pv != gv:
            return False
    return True


def score_bfcl_mini(
    cases: list[tuple[ExpectedCall | None, ExpectedCall | None]],
) -> dict[str, float]:
    """
    迷你 BFCL：把用例分成「该调用」与「该拒调」两类准确率。

    Args:
        cases: ``(pred, gold)`` 列表。

    Returns:
        metrics: ``call_acc`` / ``hallucination_acc`` / ``overall``。
    """
    call_ok = call_n = hall_ok = hall_n = 0
    for pred, gold in cases:
        if gold is None:
            hall_n += 1
            hall_ok += int(ast_match(pred, gold))
        else:
            call_n += 1
            call_ok += int(ast_match(pred, gold))
    call_acc = call_ok / call_n if call_n else 0.0
    hall_acc = hall_ok / hall_n if hall_n else 0.0
    # 对应笔记：Non-Live + Hallucination 各 10% 的玩具加权
    overall = 0.5 * call_acc + 0.5 * hall_acc
    return {
        "call_acc": call_acc,
        "hallucination_acc": hall_acc,
        "overall": overall,
    }


print("toy Toolformer + schema + BFCL-mini ready")


## 2. 玩具示例：过滤好坏候选、校验失败观察、拒调打分


In [ ]:
def demo_schema_validation() -> None:
    """坏类型 / 缺字段 → 结构化观察；纠正后成功。"""
    bad = dispatch_validated("unit_convert", {"value": "not-a-number", "from_unit": "m"})
    print("=== validation error ===")
    print(bad)
    assert "validation_error" in bad

    good = dispatch_validated(
        "unit_convert",
        {"value": "2000", "from_unit": "m", "to_unit": "km"},
    )
    print("=== after coerce ===")
    print(good)
    assert good.startswith("2")


def demo_hallucinated_tool() -> None:
    """臆造工具名 → hallucinated_tool 观察（BFCL Hallucination 精神）。"""
    obs = dispatch_validated("web_search_pro", {"q": "weather"})
    print("\n=== hallucinated tool ===")
    print(obs)
    assert "hallucinated_tool" in obs


def demo_toolformer_filter() -> None:
    """好候选降 loss 被保留；坏候选被丢掉。"""
    corpus = [
        "three plus five equals eight because 3 + 5 = 8",
        "the length 2000 m which is 2 km on the map",
        "distance conversion meters kilometers",
        "arithmetic sum addition equals",
    ]
    lm = TinyBagLM()
    lm.fit(corpus)

    text_math = "Please note that 3 + 5 = 8 in this textbook example."
    kept_math = toolformer_filter(text_math, lm, min_improvement=0.01)
    print("\n=== toolformer math ===")
    for ex in kept_math:
        print(
            f"Δloss={ex.loss_without - ex.loss_with:.4f} | {ex.call_text}"
        )
    assert any("3+5" in ex.call_text for ex in kept_math)
    assert all("3-5" not in ex.call_text for ex in kept_math)

    text_unit = "The distance is 2000 m which is 2 km exactly."
    kept_unit = toolformer_filter(text_unit, lm, min_improvement=0.01)
    print("\n=== toolformer unit ===")
    for ex in kept_unit:
        print(
            f"Δloss={ex.loss_without - ex.loss_with:.4f} | {ex.call_text}"
        )
        assert any("unit_convert" in ex.call_text and "2" in ex.tool_result for ex in kept_unit)


def demo_bfcl_mini() -> None:
    """AST 匹配 + 拒调。"""
    cases: list[tuple[ExpectedCall | None, ExpectedCall | None]] = [
        (
            ExpectedCall("calculator", {"expr": "3+5"}),
            ExpectedCall("calculator", {"expr": "3+5"}),
        ),
        (
            ExpectedCall("calculator", {"expr": "3-5"}),
            ExpectedCall("calculator", {"expr": "3+5"}),
        ),
        (None, None),  # 闲聊应拒调
        (ExpectedCall("web_search", {"q": "hi"}), None),  # 幻觉调用
    ]
    metrics = score_bfcl_mini(cases)
    print("\n=== bfcl-mini ===")
    print(metrics)
    assert metrics["call_acc"] == 0.5
    assert metrics["hallucination_acc"] == 0.5


demo_schema_validation()
demo_hallucinated_tool()
demo_toolformer_filter()
demo_bfcl_mini()
print("\nTOY DEMOS OK")


## 3. PyTorch：下一 token 损失过滤 + 工具选择头

用极小字符级 LM 复现 Toolformer 的「有工具上下文 vs 无工具」NLL 比较；再用线性头学「该不该调 / 调哪个」（Non-Live + Hallucination 玩具版）。


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class CharLM(nn.Module):
    """字符级 LSTM 语言模型（教学规模）。"""

    def __init__(self, vocab_size: int, embed_dim: int = 32, hidden: int = 64) -> None:
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim)
        self.rnn = nn.LSTM(embed_dim, hidden, batch_first=True)
        self.proj = nn.Linear(hidden, vocab_size)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: ``(B, T)`` 字符 id。

        Returns:
            logits: ``(B, T, V)`` 下一字符分布。
        """
        h, _ = self.rnn(self.embed(x))
        return self.proj(h)


class CharVocab:
    """字符词表。"""

    def __init__(self, texts: list[str]) -> None:
        chars = sorted({c for t in texts for c in t})
        self.stoi: dict[str, int] = {c: i for i, c in enumerate(chars)}
        self.itos: list[str] = chars

    def encode(self, text: str) -> torch.Tensor:
        """
        Args:
            text: 字符串。

        Returns:
            ids: ``(T,)`` long tensor；未知字符跳过。
        """
        ids = [self.stoi[c] for c in text if c in self.stoi]
        return torch.tensor(ids, dtype=torch.long)

    @property
    def size(self) -> int:
        """
        Returns:
            n: 词表大小。
        """
        return len(self.itos)


def train_char_lm(
    corpus: list[str],
    *,
    steps: int = 400,
    lr: float = 0.01,
) -> tuple[CharLM, CharVocab]:
    """
    在拼接语料上训练字符 LM。

    Args:
        corpus: 文本列表。
        steps: 优化步数。
        lr: 学习率。

    Returns:
        model: 训练后的 ``CharLM``。
        vocab: 对应词表。
    """
    vocab = CharVocab(corpus)
    data = vocab.encode("\n".join(corpus))
    model = CharLM(vocab.size)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    model.train()
    for _ in range(steps):
        if data.numel() < 4:
            break
        # 随机切片
        max_start = max(data.numel() - 32, 1)
        start = int(torch.randint(0, max_start, (1,)).item())
        chunk = data[start : start + 32]
        if chunk.numel() < 3:
            continue
        x = chunk[:-1].unsqueeze(0)
        y = chunk[1:].unsqueeze(0)
        logits = model(x)
        loss = F.cross_entropy(logits.reshape(-1, vocab.size), y.reshape(-1))
        opt.zero_grad()
        loss.backward()
        opt.step()
    model.eval()
    return model, vocab


@torch.no_grad()
def sequence_nll(model: CharLM, vocab: CharVocab, context: str, target: str) -> float:
    """
    在 ``context`` 编码后，对 ``target`` 逐字符累加 NLL（平均）。

    若某字符已在 ``context`` 中出现，额外给予 bonus（模拟工具结果降低后续不确定度）。

    Args:
        model: 字符 LM。
        vocab: 词表。
        context: 前缀（可含 API 结果文本）。
        target: 后续真实文本。

    Returns:
        nll: 平均负对数似然；过短则 ``inf``。
    """
    full = vocab.encode(context + target)
    ctx_ids = vocab.encode(context)
    ctx_len = ctx_ids.numel()
    if full.numel() < 2 or ctx_len >= full.numel():
        return float("inf")
    x = full[:-1].unsqueeze(0)
    y = full[1:]
    logits = model(x).squeeze(0)
    start = max(ctx_len - 1, 0)
    if start >= y.numel():
        return float("inf")
    logp = F.log_softmax(logits[start:], dim=-1)
    ctx_set = set(int(t) for t in ctx_ids.tolist())
    nll_sum = 0.0
    count = 0
    for i, tok in enumerate(y[start:]):
        tid = int(tok.item())
        nll = float(-logp[i, tid].item())
        if tid in ctx_set:
            nll = min(nll, 0.2)  # 结果字符已观测
        nll_sum += nll
        count += 1
    return nll_sum / max(count, 1)


def pytorch_toolformer_keep(
    model: CharLM,
    vocab: CharVocab,
    prefix: str,
    suffix: str,
    call_text: str,
    *,
    min_improvement: float = 0.02,
) -> bool:
    """
    Args:
        model: 字符 LM。
        vocab: 词表。
        prefix: 插入点前文本。
        suffix: 插入点后真实续写。
        call_text: ``[API:...]`` 片段。
        min_improvement: 保留阈值。

    Returns:
        keep: 插入后 NLL 是否足够下降。
    """
    wo = sequence_nll(model, vocab, prefix, suffix)
    wi = sequence_nll(model, vocab, prefix + call_text, suffix)
    return (wo - wi) >= min_improvement


TOOL_LABELS: list[str] = ["none", "calculator", "unit_convert"]


class ToolPolicyNet(nn.Module):
    """词袋 → 工具类别（含 none=拒调）。"""

    def __init__(self, n_words: int, n_tools: int = len(TOOL_LABELS)) -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_words, 32),
            nn.ReLU(),
            nn.Linear(32, n_tools),
        )

    def forward(self, bag: torch.Tensor) -> torch.Tensor:
        """
        Args:
            bag: ``(B, V)`` 或 ``(V,)``。

        Returns:
            logits: ``(B, C)`` 或 ``(C,)``。
        """
        single = bag.ndim == 1
        if single:
            bag = bag.unsqueeze(0)
        logits = self.net(bag)
        return logits.squeeze(0) if single else logits


WORD_VOCAB_TU: list[str] = [
    "calculate",
    "plus",
    "sum",
    "convert",
    "meters",
    "km",
    "weather",
    "poem",
    "hello",
    "math",
    "unit",
]


def bag_of_words(text: str) -> torch.Tensor:
    """
    Args:
        text: 用户请求。

    Returns:
        bag: ``(V,)`` 多热。
    """
    toks = set(re.findall(r"[a-z]+", text.lower()))
    x = torch.zeros(len(WORD_VOCAB_TU), dtype=torch.float32)
    for i, w in enumerate(WORD_VOCAB_TU):
        if w in toks:
            x[i] = 1.0
    return x


def train_tool_policy(steps: int = 300, lr: float = 0.05) -> ToolPolicyNet:
    """
    Args:
        steps: 步数。
        lr: 学习率。

    Returns:
        model: 工具策略网。
    """
    data: list[tuple[str, str]] = [
        ("calculate 3 plus 5", "calculator"),
        ("math sum please", "calculator"),
        ("convert 2000 meters to km", "unit_convert"),
        ("unit convert meters", "unit_convert"),
        ("write a poem", "none"),
        ("hello there", "none"),
        ("what is the weather", "none"),
    ]
    model = ToolPolicyNet(len(WORD_VOCAB_TU))
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    model.train()
    for _ in range(steps):
        loss = torch.tensor(0.0)
        for q, label in data:
            logits = model(bag_of_words(q))
            y = torch.tensor(TOOL_LABELS.index(label))
            loss = loss + F.cross_entropy(logits.unsqueeze(0), y.unsqueeze(0))
        loss = loss / len(data)
        opt.zero_grad()
        loss.backward()
        opt.step()
    model.eval()
    return model


@torch.no_grad()
def predict_tool(model: ToolPolicyNet, question: str) -> str:
    """
    Args:
        model: 策略网。
        question: 用户问题。

    Returns:
        label: ``none`` / ``calculator`` / ``unit_convert``。
    """
    logits = model(bag_of_words(question))
    return TOOL_LABELS[int(torch.argmax(logits).item())]


def demo_pytorch_toolformer_and_policy() -> None:
    """训练 CharLM 过滤 + ToolPolicy 拒调。"""
    torch.manual_seed(0)
    corpus = [
        "3 + 5 = 8 is basic arithmetic sum",
        "2000 m which is 2 km unit convert meters",
        "please note the textbook example equals",
        "distance conversion kilometers",
    ]
    lm, vocab = train_char_lm(corpus, steps=500)
    prefix = "Please note that "
    suffix = "3 + 5 = 8 is basic arithmetic sum"
    good_call = format_api_call("calculator", {"expr": "3+5"}, "8")
    bad_call = format_api_call("calculator", {"expr": "3-5"}, "-2")
    nll_wo = sequence_nll(lm, vocab, prefix, suffix)
    nll_good = sequence_nll(lm, vocab, prefix + good_call, suffix)
    nll_bad = sequence_nll(lm, vocab, prefix + bad_call, suffix)
    keep_good = pytorch_toolformer_keep(lm, vocab, prefix, suffix, good_call, min_improvement=0.01)
    keep_bad = pytorch_toolformer_keep(lm, vocab, prefix, suffix, bad_call, min_improvement=0.01)
    print("=== pytorch toolformer NLL filter ===")
    print(f"nll_wo={nll_wo:.4f} nll_good={nll_good:.4f} nll_bad={nll_bad:.4f}")
    print("keep_good", keep_good, "keep_bad", keep_bad)
    assert nll_good < nll_wo
    assert nll_good <= nll_bad + 1e-6
    assert keep_good

    policy = train_tool_policy()
    print("\n=== tool policy ===")
    for q in ["calculate 3 plus 5", "convert meters to km", "write a poem hello"]:
        print(q, "->", predict_tool(policy, q))
    assert predict_tool(policy, "calculate 3 plus 5") == "calculator"
    assert predict_tool(policy, "write a poem hello") == "none"
    print("PYTORCH DEMO OK")


demo_pytorch_toolformer_and_policy()


## 4. 生产级：LangChain Function Calling + DeepSeek

原生 ``bind_tools`` / ``create_agent``：Schema 来自工具签名；外层再包一层校验观察。  
覆盖笔记里的单轮调用、校验重试、以及「无关请求拒调」（Hallucination）。需 ``DEEPSEEK_API_KEY``。


In [ ]:
import json
import os
import sys
from pathlib import Path
from typing import Any, Callable

from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain_core.messages import AIMessage, BaseMessage, HumanMessage, ToolMessage
from langchain_core.tools import StructuredTool
from pydantic import BaseModel, Field, ValidationError

sys.path.append(str(Path("../../00_Common").resolve()))
from user_tools import load_project_env  # noqa: E402

load_project_env()

MODEL = "deepseek:deepseek-v4-flash"


class CalculatorArgs(BaseModel):
    """calculator 参数 Schema。"""

    expr: str = Field(description="Arithmetic expression like '(3+5)*2'")


class UnitConvertArgs(BaseModel):
    """unit_convert 参数 Schema。"""

    value: float = Field(description="Numeric magnitude")
    from_unit: str = Field(description="Source unit: m or km")
    to_unit: str = Field(description="Target unit: m or km")

    def model_post_init(self, __context: Any) -> None:
        """枚举约束（失败时由外层转成观察）。"""
        allowed = {"m", "km"}
        if self.from_unit not in allowed or self.to_unit not in allowed:
            raise ValueError("from_unit/to_unit must be 'm' or 'km'")


def _calculator_impl(expr: str) -> str:
    """
    Args:
        expr: 算术表达式。

    Returns:
        result: 计算结果。
    """
    return calculator(expr)


def _unit_convert_impl(value: float, from_unit: str, to_unit: str) -> str:
    """
    Args:
        value: 数值。
        from_unit: ``m``/``km``。
        to_unit: ``m``/``km``。

    Returns:
        result: 转换结果。
    """
    return unit_convert(value, from_unit, to_unit)


def make_validated_tool(
    name: str,
    description: str,
    args_model: type[BaseModel],
    impl: Callable[..., str],
) -> StructuredTool:
    """
    把 Pydantic Schema + 实现封成 LangChain Tool；校验失败返回 JSON 观察。

    Args:
        name: 工具名。
        description: 何时使用。
        args_model: Pydantic 参数模型。
        impl: 实际执行函数。

    Returns:
        tool: ``StructuredTool``。
    """

    def _run(**kwargs: Any) -> str:
        try:
            parsed = args_model(**kwargs)
        except (ValidationError, ValueError) as e:
            return json.dumps(
                {
                    "status": "validation_error",
                    "message": str(e),
                    "hint": "Fix args and retry.",
                },
                ensure_ascii=False,
            )
        data = parsed.model_dump()
        return impl(**data)

    return StructuredTool.from_function(
        func=_run,
        name=name,
        description=description,
        args_schema=args_model,
    )


calculator_lc = make_validated_tool(
    "calculator",
    "Evaluate arithmetic. Use only when a numeric expression must be computed.",
    CalculatorArgs,
    _calculator_impl,
)
unit_convert_lc = make_validated_tool(
    "unit_convert",
    "Convert between meters (m) and kilometers (km).",
    UnitConvertArgs,
    _unit_convert_impl,
)


def get_llm(*, temperature: float = 0.0) -> Any:
    """
    Returns:
        llm: DeepSeek chat model。
    """
    if not os.getenv("DEEPSEEK_API_KEY"):
        raise RuntimeError("DEEPSEEK_API_KEY missing; copy .env.example → .env")
    return init_chat_model(
        MODEL,
        temperature=temperature,
        extra_body={"thinking": {"type": "disabled"}},
    )


def build_tool_agent() -> Any:
    """
    Returns:
        agent: LangGraph ``create_agent`` 图。
    """
    llm = get_llm()
    return create_agent(
        llm,
        tools=[calculator_lc, unit_convert_lc],
        system_prompt=(
            "You are a careful function-calling agent. "
            "Only use calculator or unit_convert when they clearly help. "
            "If the user request is unrelated (chitchat, poetry, weather with no tool), "
            "do NOT call tools — answer briefly in Chinese. "
            "If a tool returns validation_error JSON, fix args and retry once."
        ),
    )


def format_fc_messages(messages: list[BaseMessage]) -> str:
    """
    Args:
        messages: agent 轨迹。

    Returns:
        trace: 可读多行文本。
    """
    lines: list[str] = []
    for i, m in enumerate(messages):
        if isinstance(m, HumanMessage):
            lines.append(f"{i:02d} USER        {m.content}")
        elif isinstance(m, AIMessage):
            if m.tool_calls:
                for tc in m.tool_calls:
                    lines.append(f"{i:02d} TOOL_CALL   {tc.get('name')}({tc.get('args')})")
            if m.content:
                lines.append(f"{i:02d} ASSISTANT   {m.content}")
        elif isinstance(m, ToolMessage):
            lines.append(f"{i:02d} OBS         [{m.name}] {m.content}")
        else:
            lines.append(f"{i:02d} {type(m).__name__:<11} {getattr(m, 'content', m)}")
    return "\n".join(lines)


def run_tool_agent(question: str) -> dict[str, Any]:
    """
    Args:
        question: 用户问题。

    Returns:
        result: ``invoke`` 返回值（含 ``messages``）。
    """
    agent = build_tool_agent()
    return agent.invoke({"messages": [HumanMessage(content=question)]})


def count_tool_calls(messages: list[BaseMessage]) -> int:
    """
    Args:
        messages: 轨迹。

    Returns:
        n: AIMessage 上的 tool_calls 总数。
    """
    n = 0
    for m in messages:
        if isinstance(m, AIMessage) and m.tool_calls:
            n += len(m.tool_calls)
    return n


print(f"LangChain function-calling ready | {MODEL}")


## 5. 生产示例：单轮调用、拒调、（可选）校验修复


In [ ]:
def demo_deepseek_function_calling() -> None:
    """真实 API：算术调用 + 闲聊拒调。"""
    if not os.getenv("DEEPSEEK_API_KEY"):
        print("SKIP production demo: DEEPSEEK_API_KEY missing")
        return

    q_math = "请用工具计算 (3+5)*2，只给最终数字。"
    r1 = run_tool_agent(q_math)
    print("=== math with tool ===")
    print(format_fc_messages(r1["messages"]))
    assert count_tool_calls(r1["messages"]) >= 1

    q_chat = "写一句短诗赞美春天，不要调用任何工具。"
    r2 = run_tool_agent(q_chat)
    print("\n=== chitchat should refuse tools ===")
    print(format_fc_messages(r2["messages"]))
    # Hallucination：应尽量 0 次工具调用
    print("tool_calls =", count_tool_calls(r2["messages"]))

    q_unit = "把 2000 米转成公里，用 unit_convert。"
    r3 = run_tool_agent(q_unit)
    print("\n=== unit convert ===")
    print(format_fc_messages(r3["messages"]))
    assert count_tool_calls(r3["messages"]) >= 1
    print("\nPRODUCTION DEMO OK")


demo_deepseek_function_calling()
